# S3.6 — Reading Semi-Structured Data
**Date completed:** September 2026  
**Status:** Completed  
**Key functions:** json_tuple, from_json, to_json, malformed JSON handling

In [0]:
# ============================================================
# S3.6 — Cell 2
# Three Ways to Extract Data from a JSON String Column
#
# Goal:
# 1. Compare json_tuple(), get_json_object(), and from_json().
# 2. Understand when each method is useful.
# 3. Prove the output data-type differences.
# 4. Observe JSON NULL handling.
# 5. Validate row counts and output schemas.
#
# Environment: Databricks Serverless
# Storage writes: None
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

print("=== S3.6: THREE WAYS TO READ A JSON STRING COLUMN ===")


# ------------------------------------------------------------
# Step 1 — Create Raw JSON String Data
#
# payload is StringType.
#
# This represents scenarios such as:
# - API response stored as text
# - Raw event payload
# - Decoded JSON message from a streaming source
#
# Record 5 deliberately contains JSON null values.
# ------------------------------------------------------------

raw_data = [
    (
        1,
        '{"customer_id": 201, "city": "Delhi", '
        '"order_value": 2500.0, "status": "delivered"}'
    ),
    (
        2,
        '{"customer_id": 202, "city": "Mumbai", '
        '"order_value": 1800.0, "status": "pending"}'
    ),
    (
        3,
        '{"customer_id": 203, "city": "Pune", '
        '"order_value": 3200.0, "status": "delivered"}'
    ),
    (
        4,
        '{"customer_id": 204, "city": "Chennai", '
        '"order_value": 4100.0, "status": "cancelled"}'
    ),
    (
        5,
        '{"customer_id": 205, "city": null, '
        '"order_value": null, "status": "pending"}'
    )
]

df_raw = spark.createDataFrame(
    raw_data,
    ["event_id", "payload"]
)

print("\n=== STEP 1: RAW JSON STRING DATA ===")

df_raw.printSchema()

df_raw.show(
    truncate=False
)


# ------------------------------------------------------------
# Step 2 — Method 1: json_tuple()
#
# json_tuple() extracts multiple TOP-LEVEL JSON fields
# in one function call.
#
# Important:
# - No schema is supplied.
# - Extracted values are returned as StringType.
# - Designed for top-level field extraction.
# ------------------------------------------------------------

print("\n=== STEP 2: json_tuple() ===")

df_tuple = df_raw.select(

    F.col("event_id"),

    F.json_tuple(
        F.col("payload"),
        "customer_id",
        "city",
        "order_value",
        "status"
    ).alias(
        "customer_id",
        "city",
        "order_value",
        "status"
    )
)

print("\nSchema after json_tuple():")

df_tuple.printSchema()

print("\nData after json_tuple():")

df_tuple.show(
    truncate=False
)


# ------------------------------------------------------------
# Step 3 — Method 2: get_json_object()
#
# get_json_object() extracts one JSON path per function call.
#
# Important:
# - No schema is supplied.
# - Extracted values are returned as StringType.
# - Supports JSON path expressions such as:
#
#   $.customer_id
#   $.customer.address.city
#
# Multiple function calls can be used in the same select().
# ------------------------------------------------------------

print("\n=== STEP 3: get_json_object() ===")

df_gjo = df_raw.select(

    F.col("event_id"),

    F.get_json_object(
        F.col("payload"),
        "$.customer_id"
    ).alias("customer_id"),

    F.get_json_object(
        F.col("payload"),
        "$.city"
    ).alias("city"),

    F.get_json_object(
        F.col("payload"),
        "$.order_value"
    ).alias("order_value"),

    F.get_json_object(
        F.col("payload"),
        "$.status"
    ).alias("status")
)

print("\nSchema after get_json_object():")

df_gjo.printSchema()

print("\nData after get_json_object():")

df_gjo.show(
    truncate=False
)


# ------------------------------------------------------------
# Step 4 — Define Explicit JSON Schema
#
# from_json() uses this schema to parse the JSON text
# into typed Spark fields.
# ------------------------------------------------------------

json_schema = StructType([

    StructField(
        "customer_id",
        IntegerType(),
        True
    ),

    StructField(
        "city",
        StringType(),
        True
    ),

    StructField(
        "order_value",
        DoubleType(),
        True
    ),

    StructField(
        "status",
        StringType(),
        True
    )
])


# ------------------------------------------------------------
# Step 5 — Method 3: from_json()
#
# from_json():
# - Parses JSON text.
# - Requires a schema.
# - Produces a StructType column.
# - Applies schema-defined data types.
#
# Supplying a schema controls parsing types,
# but does NOT automatically guarantee data quality.
# ------------------------------------------------------------

print("\n=== STEP 5: from_json() ===")

df_struct = df_raw.select(

    F.col("event_id"),

    F.from_json(
        F.col("payload"),
        json_schema
    ).alias("data")
)

print("\nSchema immediately after from_json():")

df_struct.printSchema()

print("\nStructured data:")

df_struct.show(
    truncate=False
)


# ------------------------------------------------------------
# Step 6 — Flatten the StructType
#
# Nested struct fields are accessed with dot notation.
# ------------------------------------------------------------

print("\n=== STEP 6: FLATTEN from_json() RESULT ===")

df_parsed = df_struct.select(

    F.col("event_id"),

    F.col(
        "data.customer_id"
    ).alias("customer_id"),

    F.col(
        "data.city"
    ).alias("city"),

    F.col(
        "data.order_value"
    ).alias("order_value"),

    F.col(
        "data.status"
    ).alias("status")
)

print("\nFlattened schema:")

df_parsed.printSchema()

print("\nFlattened data:")

df_parsed.show(
    truncate=False
)


# ------------------------------------------------------------
# Step 7 — Compare the Three Methods
# ------------------------------------------------------------

print("\n=== STEP 7: METHOD COMPARISON ===")

print(
    f"{'Method':<22}"
    f"{'Schema Needed':<18}"
    f"{'Output Types':<24}"
    f"{'Typical Use'}"
)

print("-" * 105)

comparison_rows = [
    (
        "json_tuple()",
        "No",
        "STRING",
        "Multiple top-level fields"
    ),
    (
        "get_json_object()",
        "No",
        "STRING",
        "Fields using JSON paths"
    ),
    (
        "from_json()",
        "Yes",
        "Schema-defined types",
        "Structured / nested JSON parsing"
    )
]

for method, schema_needed, output_type, typical_use in comparison_rows:

    print(
        f"{method:<22}"
        f"{schema_needed:<18}"
        f"{output_type:<24}"
        f"{typical_use}"
    )


# ------------------------------------------------------------
# Step 8 — Validate Record Counts
#
# All three methods should preserve the five source events
# in this learning example.
#
# These expected values are specific to this sample dataset.
# ------------------------------------------------------------

print("\n=== STEP 8: RECORD COUNT VALIDATION ===")

source_count = df_raw.count()
tuple_count = df_tuple.count()
gjo_count = df_gjo.count()
parsed_count = df_parsed.count()

print(f"Source records          : {source_count}")
print(f"json_tuple() records    : {tuple_count}")
print(f"get_json_object records : {gjo_count}")
print(f"from_json() records     : {parsed_count}")

assert source_count == 5, (
    "Unexpected source record count"
)

assert tuple_count == source_count, (
    "json_tuple() record-count mismatch"
)

assert gjo_count == source_count, (
    "get_json_object() record-count mismatch"
)

assert parsed_count == source_count, (
    "from_json() record-count mismatch"
)

print("\nRecord-count validation passed.")


# ------------------------------------------------------------
# Step 9 — Validate Output Data Types
#
# json_tuple() and get_json_object() return strings.
#
# from_json() applies the types from json_schema.
# ------------------------------------------------------------

print("\n=== STEP 9: DATA TYPE VALIDATION ===")

assert isinstance(
    df_tuple.schema["customer_id"].dataType,
    StringType
), "json_tuple() customer_id should be StringType"

assert isinstance(
    df_tuple.schema["order_value"].dataType,
    StringType
), "json_tuple() order_value should be StringType"


assert isinstance(
    df_gjo.schema["customer_id"].dataType,
    StringType
), "get_json_object() customer_id should be StringType"

assert isinstance(
    df_gjo.schema["order_value"].dataType,
    StringType
), "get_json_object() order_value should be StringType"


assert isinstance(
    df_parsed.schema["customer_id"].dataType,
    IntegerType
), "from_json() customer_id should be IntegerType"

assert isinstance(
    df_parsed.schema["order_value"].dataType,
    DoubleType
), "from_json() order_value should be DoubleType"

print("Data-type validation passed.")


# ------------------------------------------------------------
# Step 10 — Inspect JSON NULL Handling
#
# Record 5 contains:
#
# "city": null
# "order_value": null
#
# JSON null becomes SQL NULL after extraction/parsing.
# ------------------------------------------------------------

print("\n=== STEP 10: JSON NULL HANDLING ===")

print("from_json() output for event_id = 5:")

df_parsed.filter(
    F.col("event_id") == 5
).show(
    truncate=False
)


# ------------------------------------------------------------
# Step 11 — Final Learning Summary
# ------------------------------------------------------------

print("\n=== FINAL LEARNING SUMMARY ===")

print("""
json_tuple():
- Extracts multiple top-level JSON fields in one call.
- Does not require an explicit schema.
- Returns extracted values as StringType.

get_json_object():
- Extracts one JSON path per function call.
- Does not require an explicit schema.
- Supports JSON-path expressions.
- Returns extracted values as StringType.

from_json():
- Parses JSON text using a supplied schema.
- Produces structured Spark data.
- Applies schema-defined data types.
- Is useful for structured and nested JSON processing.

Important:
- JSON null values become SQL NULL.
- Schema definition does not automatically guarantee data quality.
- Choose the JSON function based on the structure and required output.
- Malformed JSON handling will be tested separately.
""")

print("=== CELL 2 COMPLETED ===")

In [0]:
# ============================================================
# S3.6 — Cell 3
# to_json() and Malformed JSON Handling
#
# Goal:
# 1. Convert StructType data to JSON using to_json().
# 2. Convert MapType data to JSON using to_json().
# 3. Parse valid and malformed JSON safely.
# 4. Distinguish:
#       - Valid JSON
#       - Valid JSON containing NULL
#       - Malformed JSON
#       - Empty payload
# 5. Quarantine parsing failures correctly.
#
# Environment: Databricks Serverless
# Storage writes: None
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    MapType
)

print("=== S3.6: TO_JSON AND MALFORMED JSON HANDLING ===")

# ------------------------------------------------------------
# Step 1 — Create Structured Data
# ------------------------------------------------------------

df_struct = spark.createDataFrame(
    [
        (1001, "Delhi", 2500.0, "delivered"),
        (1002, "Mumbai", 1800.0, "pending"),
        (1003, "Pune",   3200.0, "cancelled")
    ],

    ["order_id", "city", "order_value", "status"]

)

print("\n=== STEP 1: ORIGINAL STRUCTURED DATA ===")

df_struct.printSchema()
df_struct.show(truncate=False)

# ------------------------------------------------------------
# Step 2 — Convert Struct → JSON String
#
# struct() groups multiple columns into a StructType.
# to_json() serializes that structure into JSON text.
#
# Typical uses:
# - Event payload creation
# - API responses
# - Kafka message values
# - Downstream integrations
# ------------------------------------------------------------

print("\n=== STEP 2: STRUCT → JSON ===")

df_json_out = df_struct.select(
    F.col("order_id"),

    F.to_json(
        F.struct(
            F.col("city"),
            F.col("order_value"),
            F.col("status")).alias("payload_json")
    )
)

df_json_out.printSchema()
df_json_out.show(truncate=False)

# ------------------------------------------------------------
# Step 3 — Convert MapType → JSON String
#
# to_json() can also serialize MapType values.
# ------------------------------------------------------------

print("\n=== STEP 3: MAP → JSON ===")

map_data = [
    (1001,{"source": "website", "channel": "online"}),
    (1002,{"source": "mobile", "channel": "app"})
]

map_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("metadata", 
                MapType(
                    StringType(), 
                    StringType(),
                    valueContainsNull=True), True)
])

df_map = spark.createDataFrame(
    map_data,
    schema=map_schema
)

print("\nOriginal MapType data:")

df_map.show(truncate=False)

df_map_json = df_map.select(
    F.col("order_id"),

    F.to_json("metadata").alias("metadata_json"))

print("\nMapType converted to JSON:")

df_map_json.show(truncate=False)

# ------------------------------------------------------------
# Step 4 — Create Valid and Invalid JSON Payloads
#
# Different scenarios:
#
# Event 1 → Valid JSON
# Event 2 → Malformed JSON (missing closing brace)
# Event 3 → Completely invalid text
# Event 4 → Valid JSON containing NULL
# Event 5 → Empty payload
# Event 6 → Valid JSON but missing order_value
#
# Missing fields and NULL values are NOT automatically
# malformed JSON.
# ------------------------------------------------------------

print("\n=== STEP 4: TEST JSON PAYLOADS ===")

malformed_data = [
        (1,'{"customer_id": 201, ' '"city": "Delhi", ' '"order_value": 2500.0}'),
        (2,'{"customer_id": 202, ' '"city": "Mumbai"'),
        (3,'NOT JSON AT ALL'),
        (4, '{"customer_id": 204, ' '"city": null, ' '"order_value": null}'),
        (5, ''),
        (6, '{"customer_id": 206, ' '"city": "Pune"}')
]

df_malformed = spark.createDataFrame(
    malformed_data,["event_id", "payload"]
)

df_malformed.show(truncate = False)

# ------------------------------------------------------------
# Step 5 — Define JSON Parsing Schema
#
# _corrupt_record is included so PERMISSIVE mode can
# retain malformed source text for investigation.
# ------------------------------------------------------------

json_schema = StructType([

    StructField("customer_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("order_value", DoubleType(), True),
    StructField("_corrupt_record", StringType(), True)
])

# ------------------------------------------------------------
# Step 6 — Parse Using PERMISSIVE Mode
#
# PERMISSIVE is suitable when we want processing to continue
# while retaining malformed records for investigation.
#
# columnNameOfCorruptRecord identifies the field that stores
# the malformed JSON text.
# ------------------------------------------------------------

print("\n=== STEP 6: PARSE JSON — PERMISSIVE MODE ===")

df_parsed = df_malformed.withColumn(

    "data",

    F.from_json(
        F.col("payload"),
        json_schema,
        {
            "mode": "PERMISSIVE",
            "columnNameOfCorruptRecord": "_corrupt_record"
        }
    )
)

df_parsed.printSchema()

df_parsed.show(
    truncate=False
)


# ------------------------------------------------------------
# Step 7 — Flatten Parsed Fields
# ------------------------------------------------------------
print("\n=== STEP 7: FLATTEN PARSED JSON ===")

df_result = df_parsed.select(

    F.col("event_id"),
    F.col("payload"),
    F.col("data.customer_id").alias("customer_id"),
    F.col("data.city").alias("city"),
    F.col("data.order_value").alias("order_value"),
    F.col("data._corrupt_record").alias("_corrupt_record")
)

df_result.show(truncate=False)

# ------------------------------------------------------------
# Step 8 — Assign Parsing Status
#
# Important:
#
# Parsing quality and business-data quality are DIFFERENT.
#
# Example:
# customer_id = NULL might be bad business data,
# but it does NOT automatically mean malformed JSON.
# ------------------------------------------------------------

print("\n=== STEP 8: CLASSIFY PARSING STATUS ===")

df_status = df_result.withColumn(
    "parse_status",
    F.when(F.col("payload").isNull(),F.lit("NULL_PAYLOAD"))
    .when(F.trim(F.col("payload")) == "",F.lit("EMPTY_PAYLOAD"))
    .when(F.col("_corrupt_record").isNotNull(),F.lit("MALFORMED_JSON"))
    .otherwise(F.lit("VALID_JSON"))
)

df_status.select(
    "event_id",
    "customer_id",
    "city",
    "order_value",
    "parse_status"
).show(truncate=False)

# ------------------------------------------------------------
# Step 9 — Separate Valid and Quarantine Records
#
# Quarantine is based on parsing status,
# NOT customer_id being NULL.
# ------------------------------------------------------------

print("\n=== STEP 9: QUARANTINE PATTERN ===")

df_good = df_status.filter(
    F.col("parse_status") == "VALID_JSON")

df_quarantine = df_status.filter(
    F.col("parse_status") != "VALID_JSON")

good_count = df_good.count()
quarantine_count = df_quarantine.count()

print(f"Valid JSON records : {good_count}")
print(f"Quarantine records : {quarantine_count}")

print("\n--- VALID JSON RECORDS ---")

df_good.select(
    "event_id",
    "customer_id",
    "city",
    "order_value"
).show(truncate=False)

print("\n--- QUARANTINE RECORDS ---")

df_quarantine.select(
    "event_id",
    "payload",
    "parse_status"
).show(truncate=False)


# ------------------------------------------------------------
# Step 10 — Business Validation
#
# This happens AFTER parsing.
#
# A record can contain valid JSON but still fail
# business-quality rules.
# ------------------------------------------------------------

print("\n=== STEP 10: BUSINESS VALIDATION ===")

df_business_validation = df_good.withColumn(

    "business_status",

    F.when(
        F.col("customer_id").isNull(),
        F.lit("INVALID_CUSTOMER_ID")
    ).otherwise(F.lit("VALID"))
)

df_business_validation.select(
    "event_id",
    "customer_id",
    "city",
    "order_value",
    "business_status"
).show(truncate=False)

# ------------------------------------------------------------
# Step 11 — Record Count Validation
# ------------------------------------------------------------

print("\n=== STEP 11: RECORD COUNT VALIDATION ===")

source_count = df_malformed.count()
classified_count = df_status.count()

final_count = (good_count + quarantine_count)

print(f"Source records       : {source_count}")
print(f"Classified records   : {classified_count}")
print(f"Valid JSON records   : {good_count}")
print(f"Quarantine records   : {quarantine_count}")
print(f"Reconciled records   : {final_count}")

assert classified_count == source_count, (
    "Parsing unexpectedly changed record count")

assert final_count == source_count, (
    "Good + quarantine records do not match source count")

print("\nRecord-count validation passed.")

# ------------------------------------------------------------
# Step 12 — Final Learning Summary
# ------------------------------------------------------------

print("\n=== FINAL LEARNING SUMMARY ===")

print("""
to_json():
- Serializes structured Spark data into JSON text.
- Works with StructType and MapType.
- Useful for downstream event/API payload creation.

from_json():
- Deserializes JSON text into structured Spark data.
- Can use parsing modes such as PERMISSIVE and FAILFAST.

PERMISSIVE:
- Allows processing to continue.
- Malformed input can be captured for quarantine.

Important distinctions:

VALID JSON + NULL VALUE
is different from
MALFORMED JSON.

MISSING JSON FIELD
is different from
MALFORMED JSON.

JSON PARSING VALIDATION
is different from
BUSINESS DATA-QUALITY VALIDATION.

Enterprise Pattern:

Raw Payload
      ↓
Parse JSON
      ↓
Classify Valid / Malformed
      ↓
Quarantine Parsing Failures
      ↓
Apply Business Validation
      ↓
Process Valid Records
""")

print("=== CELL 3 COMPLETED ===")


%md
# S3.6 — Reading Semi-Structured Data
**Status:** ✅ Completed  
**Environment:** Databricks Serverless

---

# 1. What We Learned

In S3.6 we learned how to work with JSON stored inside Spark STRING columns.

Main functions:

- `json_tuple()`
- `get_json_object()`
- `from_json()`
- `to_json()`

We also learned:

- Valid JSON vs malformed JSON
- JSON NULL vs missing fields
- PERMISSIVE parsing
- FAILFAST concept
- Corrupt-record handling
- Quarantine pattern
- Parsing validation vs business validation
- Record-count reconciliation

---

# 2. Final Confirmation — Which JSON Function Should I Use?

## 2.1 `json_tuple()` — Extract Multiple Top-Level Fields

### Purpose

`json_tuple()` extracts multiple simple **top-level fields** from a JSON STRING.

### Use when

- JSON is stored as a STRING column.
- You need several simple top-level fields.
- You do not need an explicit schema.
- STRING output is acceptable.
- You do not need complex nested JSON-path navigation.

### Example input

```json
{
  "customer_id": 201,
  "city": "Delhi",
  "status": "delivered"
}
```

### Example

```python
F.json_tuple(
    F.col("payload"),
    "customer_id",
    "city",
    "status"
)
```

Typical output types:

```text
customer_id → StringType
city        → StringType
status      → StringType
```

### Typical scenario

```text
Raw JSON STRING
       ↓
Need several simple top-level values
       ↓
json_tuple()
       ↓
STRING columns
```

### Real use case

Suppose an API/event payload contains 20 fields, but we only need:

```text
customer_id
city
status
```

and all three are top-level fields.

`json_tuple()` provides a simple way to extract them.

### Important limitation

`json_tuple()` is mainly for top-level fields.

For nested navigation such as:

```text
customer.address.city
```

`get_json_object()` or `from_json()` is more appropriate.

---

# 3. `get_json_object()` — Extract a Field Using JSON Path

## Purpose

`get_json_object()` extracts a value from a JSON STRING using a JSON-path expression.

### Use when

- JSON is stored as STRING.
- You need a particular field.
- You need JSON-path navigation.
- The field may be nested.
- You do not need to parse the entire payload into a Spark structure.

### Example input

```json
{
  "customer": {
    "address": {
      "city": "Delhi"
    }
  }
}
```

### Example

```python
F.get_json_object(
    F.col("payload"),
    "$.customer.address.city"
)
```

Result:

```text
Delhi
```

Output type:

```text
StringType
```

### Typical scenario

```text
JSON STRING
    ↓
Need specific nested path
    ↓
$.customer.address.city
    ↓
get_json_object()
    ↓
STRING value
```

### Important

One `get_json_object()` call extracts one JSON path.

But multiple calls can be used in the same `select()`.

Example:

```python
df.select(
    F.get_json_object(
        "payload",
        "$.customer_id"
    ).alias("customer_id"),

    F.get_json_object(
        "payload",
        "$.city"
    ).alias("city"),

    F.get_json_object(
        "payload",
        "$.status"
    ).alias("status")
)
```

### Real use case

Suppose the payload is large, but your requirement is only:

```text
customer.address.city
```

You do not necessarily need to parse the complete JSON structure.

You can directly extract:

```python
F.get_json_object(
    "payload",
    "$.customer.address.city"
)
```

---

# 4. `from_json()` — Convert JSON String into Structured Spark Data

## Purpose

`from_json()` converts JSON stored as STRING into structured Spark data using a supplied schema.

### Use when

- JSON is stored as STRING.
- You know the expected structure.
- You need proper Spark data types.
- You need nested JSON processing.
- You want structured downstream transformations.
- You are building a governed ingestion/transformation pipeline.

### Example input

```json
{
  "customer_id": 201,
  "city": "Delhi",
  "order_value": 2500.0
}
```

### Define schema

```python
json_schema = StructType([
    StructField(
        "customer_id",
        IntegerType(),
        True
    ),

    StructField(
        "city",
        StringType(),
        True
    ),

    StructField(
        "order_value",
        DoubleType(),
        True
    )
])
```

### Parse

```python
F.from_json(
    F.col("payload"),
    json_schema
)
```

### Result

```text
customer_id → IntegerType
city        → StringType
order_value → DoubleType
```

Unlike `json_tuple()` and `get_json_object()`, we now get
the data types defined by our schema.

### Typical enterprise scenario

```text
Kafka / API JSON payload
          ↓
      JSON STRING
          ↓
   from_json(schema)
          ↓
       STRUCT
          ↓
Parsing Validation
          ↓
Business Validation
          ↓
Transformation
          ↓
Silver Data
```

### Real use case

Suppose an incoming order payload contains:

```json
{
  "order_id": 1001,
  "customer": {
    "customer_id": 201,
    "city": "Delhi"
  },
  "order_value": 2500.0
}
```

If we need to process the complete order with proper Spark types,
a structured schema is more appropriate:

```python
F.from_json(
    F.col("payload"),
    order_schema
)
```

Then nested fields can be accessed using:

```python
F.col("data.customer.customer_id")
F.col("data.customer.city")
F.col("data.order_value")
```

### Important

The schema defines the **expected Spark data types**.

It does NOT automatically guarantee:

- Correct business values
- Required fields are populated
- No duplicates
- Positive order amounts
- Valid customer IDs
- Complete data quality

Those checks must be performed separately.

---

# 5. `to_json()` — Convert Structured Spark Data into JSON String

## Purpose

`to_json()` does the reverse of `from_json()`.

```text
from_json()

JSON STRING
     ↓
STRUCTURED SPARK DATA


to_json()

STRUCTURED SPARK DATA
     ↓
JSON STRING
```

### Use when

- Spark data is already structured.
- You need to send records to another system as JSON.
- You are preparing Kafka output.
- You are creating an event payload.
- You are creating an API/downstream-system payload.

### Example structured data

```text
order_id     = 1001
city         = Delhi
order_value  = 2500.0
```

### Example

```python
F.to_json(
    F.struct(
        "order_id",
        "city",
        "order_value"
    )
)
```

### Output

```json
{
  "order_id": 1001,
  "city": "Delhi",
  "order_value": 2500.0
}
```

### Typical scenario

```text
Structured Spark DataFrame
          ↓
Transformation
          ↓
to_json()
          ↓
JSON STRING
          ↓
Kafka / API / downstream system
```

### StructType example

```python
F.to_json(
    F.struct(
        "city",
        "order_value",
        "status"
    )
)
```

### MapType example

```python
F.to_json(
    F.col("metadata")
)
```

We demonstrated both StructType and MapType serialization.

---

# 6. Quick Decision Table

| Requirement | Function |
|---|---|
| Need several simple top-level fields | `json_tuple()` |
| Need a field using JSON path | `get_json_object()` |
| Need nested JSON-path extraction without full parsing | `get_json_object()` |
| Need typed structured JSON | `from_json()` |
| Need nested JSON transformation | `from_json()` |
| Need Spark Struct/Map converted back to JSON | `to_json()` |

---

# 7. Easy Way to Remember

```text
json_tuple()

JSON STRING
    ↓
Selected TOP-LEVEL fields
    ↓
STRING output
```

```text
get_json_object()

JSON STRING
    ↓
JSON PATH
    ↓
STRING output
```

```text
from_json()

JSON STRING
    ↓
SCHEMA
    ↓
STRUCTURED / TYPED SPARK DATA
```

```text
to_json()

STRUCTURED SPARK DATA
    ↓
JSON STRING
```

---

# 8. Real Enterprise Example

Suppose Kafka contains this order payload:

```json
{
  "order_id": 1001,
  "customer": {
    "customer_id": 201,
    "city": "Delhi"
  },
  "order_value": 2500.0,
  "status": "delivered"
}
```

## Requirement 1 — Need only nested customer city

Use:

```python
F.get_json_object(
    "payload",
    "$.customer.city"
)
```

Because we only need one nested path.

---

## Requirement 2 — Need several simple top-level values

Suppose we only need:

```text
order_id
order_value
status
```

Use:

```python
F.json_tuple(
    "payload",
    "order_id",
    "order_value",
    "status"
)
```

---

## Requirement 3 — Need complete payload with proper data types

Use:

```python
F.from_json(
    "payload",
    json_schema
)
```

Then perform:

```text
Validation
Transformation
Flattening
Silver processing
```

---

## Requirement 4 — Send transformed order to another Kafka topic

Use:

```python
F.to_json(
    F.struct(
        "order_id",
        "customer_id",
        "order_value",
        "status"
    )
)
```

Result:

```text
Structured Spark Data
        ↓
JSON STRING
        ↓
Kafka
```

---

# 9. Serialization vs Deserialization

## Deserialization

Converting serialized JSON text into structured Spark data.

```text
JSON STRING
     ↓
from_json()
     ↓
Spark Struct / Array / Map
```

## Serialization

Converting structured Spark data into JSON text.

```text
Spark Struct / Map
       ↓
to_json()
       ↓
JSON STRING
```

---

# 10. Valid JSON vs Malformed JSON

These cases must not be mixed.

## Valid JSON

```json
{
  "customer_id": 201,
  "city": "Delhi"
}
```

---

## Valid JSON containing NULL

```json
{
  "customer_id": 204,
  "city": null,
  "order_value": null
}
```

This JSON is still syntactically valid.

---

## Valid JSON with missing field

```json
{
  "customer_id": 206,
  "city": "Pune"
}
```

`order_value` may be missing, but the JSON itself can still be valid.

---

## Malformed JSON

```text
{"customer_id": 202, "city": "Mumbai"
```

Missing closing brace.

Another example:

```text
NOT JSON AT ALL
```

---

# 11. Critical Distinctions

```text
VALID JSON + NULL VALUE
        ≠
MALFORMED JSON
```

```text
MISSING JSON FIELD
        ≠
MALFORMED JSON
```

```text
JSON PARSING VALIDATION
        ≠
BUSINESS DATA-QUALITY VALIDATION
```

This is one of the most important lessons from S3.6.

---

# 12. PERMISSIVE Mode

In our malformed-JSON practical we used:

```python
F.from_json(
    F.col("payload"),
    json_schema,
    {
        "mode": "PERMISSIVE",
        "columnNameOfCorruptRecord": "_corrupt_record"
    }
)
```

Purpose:

- Continue processing where supported.
- Capture malformed input.
- Preserve problematic source payloads for investigation.
- Separate bad parsing records from good records.

Typical architecture:

```text
Incoming Payload
       ↓
from_json()
PERMISSIVE
       ↓
 ┌───────────────┐
 │               │
Valid        Malformed
 │               │
 ↓               ↓
Process       Quarantine
```

---

# 13. FAILFAST Mode

`FAILFAST` represents a stricter approach.

Conceptually:

```text
Malformed JSON
      ↓
FAILFAST
      ↓
Stop / Fail Processing
```

Use when the ingestion contract requires malformed input
to stop processing rather than continue.

---

# 14. Correct Quarantine Pattern

Do NOT identify malformed JSON using:

```python
customer_id.isNull()
```

Why?

Because this is valid JSON:

```json
{
  "customer_id": null
}
```

The JSON is parseable even though the business field is NULL.

Correct architecture:

```text
Raw Payload
      ↓
Check NULL / EMPTY payload
      ↓
Parse JSON
      ↓
 ┌────────────────────────────┐
 │                            │
Valid JSON              Malformed JSON
 │                            │
 ↓                            ↓
Business Validation       Quarantine
 │
 ↓
Valid Business Record
 │
 ↓
Process
```

---

# 15. Parsing Validation vs Business Validation

## Parsing Validation

Question:

**Can Spark correctly interpret the payload as JSON?**

Possible problems:

- Malformed syntax
- Completely invalid text
- Empty payload
- NULL payload

---

## Business Validation

Question:

**Does the successfully parsed record satisfy business rules?**

Examples:

```text
customer_id IS NULL
order_value < 0
quantity <= 0
unknown status
missing mandatory business field
duplicate business key
```

Therefore:

```text
VALID JSON
    ↓
can still contain
    ↓
INVALID BUSINESS DATA
```

---

# 16. Record Reconciliation

After classification:

```python
assert classified_count == source_count
```

and:

```python
assert (
    good_count + quarantine_count
    == source_count
)
```

Meaning:

```text
SOURCE RECORDS
      =
VALID JSON RECORDS
      +
QUARANTINE RECORDS
```

This proves that classification did not unexpectedly lose records.

However:

**Matching counts does not prove complete data quality.**

Additional business validations may still be required.

---

# 17. Complete Enterprise JSON Processing Pattern

```text
Source
Kafka / API / File
        ↓
Raw JSON Payload
        ↓
NULL / EMPTY Check
        ↓
Parse JSON
        ↓
 ┌──────────────────────────────┐
 │                              │
Malformed JSON               Valid JSON
 │                              │
 ↓                              ↓
Quarantine              Business Validation
                               ↓
                        Typed Structured Data
                               ↓
                         Transform / Flatten
                               ↓
                            Validate
                               ↓
                             Silver
                               ↓
                      Optional to_json()
                               ↓
                     Kafka / API / Events
```

---

# 18. Final Function Selection Cheat Sheet

```text
QUESTION:
I need multiple simple TOP-LEVEL fields.

USE:
json_tuple()
```

```text
QUESTION:
I need a specific field using a JSON path.

USE:
get_json_object()
```

```text
QUESTION:
I need the JSON converted into typed structured Spark data.

USE:
from_json()
```

```text
QUESTION:
I already have structured Spark data and need JSON output.

USE:
to_json()
```

---

# 19. Interview Questions and Answers

### Q1. What is semi-structured data?

Semi-structured data does not necessarily follow a fixed relational
row-and-column structure but contains structural information such
as keys, objects, arrays and nested fields.

JSON is a common example.

---

### Q2. What does `json_tuple()` do?

It extracts multiple top-level fields from a JSON string.

The extracted values are returned as strings.

---

### Q3. When would you use `json_tuple()`?

When several simple top-level JSON fields are required and a complete
typed JSON parse is unnecessary.

---

### Q4. What does `get_json_object()` do?

It extracts a value from JSON text using a JSON-path expression.

Example:

```python
F.get_json_object(
    "payload",
    "$.customer.address.city"
)
```

---

### Q5. What is the difference between `json_tuple()` and `get_json_object()`?

`json_tuple()` is convenient for extracting multiple top-level fields.

`get_json_object()` extracts one JSON path per function call and can
navigate nested paths.

Both return extracted values as strings.

---

### Q6. What does `from_json()` do?

It deserializes JSON text into structured Spark data using a supplied schema.

---

### Q7. Why use `from_json()`?

Use it when we need:

- Structured data
- Defined Spark data types
- Nested JSON processing
- Downstream transformations based on a known schema

---

### Q8. Does `from_json()` guarantee data quality?

No.

The schema controls parsing structure and expected types.

Business-rule validation is still required separately.

---

### Q9. What does `to_json()` do?

It serializes structured Spark data such as StructType or MapType
into a JSON string.

---

### Q10. Give a practical use case for `to_json()`.

After transforming an order in Spark, we may convert the structured
record into JSON before publishing it to Kafka or another downstream system.

---

### Q11. What is the difference between `from_json()` and `to_json()`?

```text
from_json()
JSON STRING → STRUCTURED DATA

to_json()
STRUCTURED DATA → JSON STRING
```

---

### Q12. What is serialization?

Serialization converts structured data into a transferable representation.

In this lesson:

```text
Spark Struct
    ↓
to_json()
    ↓
JSON STRING
```

---

### Q13. What is deserialization?

Deserialization converts serialized data back into structured data.

In this lesson:

```text
JSON STRING
    ↓
from_json()
    ↓
Spark Structure
```

---

### Q14. Is JSON containing NULL malformed?

No.

Example:

```json
{
  "customer_id": 201,
  "city": null
}
```

This is valid JSON.

---

### Q15. Is a missing JSON field the same as malformed JSON?

No.

JSON syntax may still be completely valid even when an expected
business field is absent.

---

### Q16. What is malformed JSON?

JSON text that does not satisfy valid JSON syntax.

Example:

```text
{"customer_id": 201, "city": "Delhi"
```

The closing brace is missing.

---

### Q17. What is PERMISSIVE mode?

PERMISSIVE parsing allows processing to continue where supported
while malformed input can be captured for investigation.

It is useful for quarantine-oriented ingestion patterns.

---

### Q18. What is FAILFAST?

FAILFAST represents strict parsing behaviour where malformed input
causes processing to fail rather than continue.

---

### Q19. Why should we not identify malformed JSON using `customer_id IS NULL`?

Because:

```json
{
  "customer_id": null
}
```

is syntactically valid JSON.

NULL business values and malformed JSON are different problems.

---

### Q20. What is a quarantine pattern?

It separates records that cannot be parsed or processed normally
from records that can continue through the pipeline.

Example:

```text
Source
  ↓
Parse
  ↓
Good ─────────→ Processing
Bad  ─────────→ Quarantine
```

---

### Q21. What is the difference between parsing validation and business validation?

Parsing validation asks:

**Can I interpret the payload correctly?**

Business validation asks:

**Is the interpreted data acceptable according to business rules?**

---

### Q22. How would you process Kafka JSON in Spark?

Typical pattern:

```text
Kafka value
      ↓
Decode to STRING
      ↓
from_json(schema)
      ↓
Parsing validation
      ↓
Quarantine malformed records
      ↓
Business validation
      ↓
Transformation
      ↓
Target
```

---

### Q23. If you need only `$.customer.city`, which function would you choose?

`get_json_object()` is a suitable option because it can directly
extract the required nested JSON path.

---

### Q24. If you need customer_id, city and status and all are top-level fields?

`json_tuple()` can extract all three top-level fields together.

---

### Q25. If you need the complete order JSON with Integer, Double, Array and Struct fields?

Use `from_json()` with an appropriately defined schema.

---

### Q26. If you need to publish the transformed record back to Kafka?

Use `to_json()` to serialize the required structured columns into JSON text.

---

# 20. Final S3.6 Understanding

We learned two main directions.

## Reading / Deserializing JSON

```text
JSON STRING
    ↓
json_tuple()
or
get_json_object()
or
from_json()
    ↓
Spark Data
```

But they serve different purposes:

```text
json_tuple()
→ Several top-level fields

get_json_object()
→ Specific JSON path

from_json()
→ Complete structured / typed representation
```

## Writing / Serializing JSON

```text
Spark Structured Data
        ↓
to_json()
        ↓
JSON STRING
```

---

# Final Learning Principle

Choose the JSON function based on the requirement.

Do not assume one function is universally better.

```text
Need simple top-level extraction?
→ json_tuple()

Need JSON path navigation?
→ get_json_object()

Need structured typed processing?
→ from_json()

Need JSON output?
→ to_json()
```

Then follow the enterprise processing sequence:

```text
Parse
  ↓
Classify
  ↓
Quarantine
  ↓
Business Validate
  ↓
Transform
  ↓
Reconcile
  ↓
Persist / Publish
```

**S3.6 — COMPLETED ✅**